# What each era actually returns

The other notebooks plot mAP. This one shows the thing mAP summarizes: for a query photo,
the top of the ranking each era's best method returns from roxford5k.

- **classic**: RootSIFT VLAD at k = 1024 — not the table's k = 4096, whose encoding peaks above
  50 GB of RAM; k = 1024 scores 0.331 / 0.162 roxford5k Medium / Hard against 0.385 / 0.195;
- **CNN**: Radenović et al.'s fine-tuned GeM ResNet101 (`gem-ft-r101`);
- **ViT**: frozen DINOv3 ViT-L/16 at the 224 protocol.

Marks follow the Medium protocol: ✓ correct (an *easy* or *hard* positive), ✗ wrong, – ignored
(*junk*: too ambiguous to score, removed from the ranking before mAP is computed — shown
here only so the strip is the ranking as returned).

**Which queries.** Fixed by a rule decided before looking, so the figure is not a
showcase: the query on which DINOv3 scores what it scores on average (its AP there is the
closest to its mAP) — representative of the best method, not its best case — and the query
on which VLAD scores what *it* scores on average.

**The grey photos.** In the first example DINOv3's top three are marked junk. They are the
same tower, photographed badly — at dusk, behind a pub sign, cut off at the edge of
the frame — and the annotators mark photos that hard to call as junk, so they count neither
for nor against. They are the right building, and DINOv3 finds it even there. Rankings are computed by `cbir.eval.runner.rank`, the step `cbir evaluate` scores, and
cached under `data/rankings/`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

sys.path.insert(0, ".")  # style.py sits beside this notebook
import style
from cbir.configs.classic import VLADConfig
from cbir.configs.presets import PRESETS
from cbir.configs.run import RunConfig
from cbir.configs.vit import ViTConfig
from cbir.data.crop import crop_query
from cbir.data.images import image_paths
from cbir.data.revisitop import download
from cbir.eval.metrics import protocol_sets, query_ap
from cbir.eval.runner import ground_truth, rank

style.use()
FIGURES = Path("..") / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
CACHE = Path("..") / "data" / "rankings"
CACHE.mkdir(parents=True, exist_ok=True)
DATASET = "roxford5k"
TOP = 8

METHODS = {
    "VLAD k=1024 (classic)": ("vlad-k1024", VLADConfig(k=1024)),
    "GeM, fine-tuned (CNN)": ("gem-ft-r101", PRESETS["gem-ft-r101"]),
    "DINOv3, frozen (ViT)": ("dinov3-vitl16", ViTConfig(model="dinov3-vitl16")),
}

## Rankings

Computed once per method and cached: VLAD re-reads the cached RootSIFT descriptors and
encodes them (minutes to tens of minutes, ~30 GB of RAM at k = 1024), the neural methods extract on the GPU (minutes).

In [ ]:
rankings = {}
for label, (name, descriptor) in METHODS.items():
    path = CACHE / f"{name}-{DATASET}.npy"
    if not path.exists():
        np.save(path, rank(RunConfig(descriptor=descriptor, dataset=DATASET)))
    rankings[label] = np.load(path)

queries, database = download(DATASET)
truth = ground_truth(queries)
query_paths, database_paths = image_paths(queries), image_paths(database)

def ap(label, q):
    positive, ignored = protocol_sets(truth[q].easy, truth[q].hard, truth[q].junk, "medium")
    return query_ap(rankings[label][q], positive, ignored) if positive else float("nan")

aps = {label: np.array([ap(label, q) for q in range(len(truth))]) for label in METHODS}
for label, values in aps.items():
    print(f"{label:30s} mAP {np.nanmean(values):.3f}")

## The figure

In [ ]:
def typical(label):
    values = aps[label]
    return int(np.nanargmin(np.abs(values - np.nanmean(values))))

# Titled by what makes each query representative: the method scores on it what it scores
# on average, so it is neither that method's showcase nor its failure.
CHOSEN = {
    "DINOv3 scores here what\nit scores on average": typical("DINOv3, frozen (ViT)"),
    "VLAD scores here what\nit scores on average": typical("VLAD k=1024 (classic)"),
}

def thumb(path, box=None, side=160):
    image = Image.open(path).convert("RGB")
    if box is not None:
        image = crop_query(image, box)
    image.thumbnail((side, side))
    return image

def status(q, index):
    positive, ignored = protocol_sets(truth[q].easy, truth[q].hard, truth[q].junk, "medium")
    if index in positive:
        return "correct", "✓"
    if index in ignored:
        return "ignored", "–"
    return "wrong", "✗"

fig_examples = plt.figure(figsize=(15, 4.2 * len(CHOSEN)))
outer = fig_examples.add_gridspec(len(CHOSEN), 1, hspace=0.22)
for block, (why, q) in enumerate(CHOSEN.items()):
    # Query, then a column of method labels, then the top of each ranking: the labels get
    # their own column so they never sit on the query photo.
    grid = outer[block].subgridspec(len(METHODS), TOP + 2, wspace=0.06, hspace=0.12,
                                    width_ratios=[1.7, 1.25] + [1] * TOP)
    query_ax = fig_examples.add_subplot(grid[:, 0])
    query_ax.imshow(thumb(query_paths[q], queries[q]["bbx"], side=240))
    query_ax.set_title(f"query {q}\n{why}", fontsize=9, color=style.INK)
    query_ax.axis("off")
    for row, label in enumerate(METHODS):
        label_ax = fig_examples.add_subplot(grid[row, 1])
        label_ax.axis("off")
        label_ax.text(0.95, 0.5, f"{label.replace(' (', chr(10) + '(')}\nAP {aps[label][q]:.2f}", ha="right", va="center",
                      fontsize=8.5, color=style.INK, transform=label_ax.transAxes)
        for col in range(TOP):
            index = int(rankings[label][q][col])
            kind, glyph = status(q, index)
            ax = fig_examples.add_subplot(grid[row, col + 2])
            ax.imshow(thumb(database_paths[index]))
            ax.set_xticks([]), ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_color(style.MATCH[kind])
                spine.set_linewidth(3)
            ax.set_title(glyph, fontsize=10, color=style.INK, pad=2)
handles = [plt.Rectangle((0, 0), 1, 1, facecolor="none", edgecolor=c, linewidth=3) for c in style.MATCH.values()]
fig_examples.legend(handles, ["✓ correct", "✗ wrong", "– ignored (junk)"], loc="upper center", ncol=3,
                    frameon=False, bbox_to_anchor=(0.5, 0.96))

## Write the docs figure

In [ ]:
# White rather than transparent, for GitHub's dark theme.
path = FIGURES / "retrieval_examples.png"
fig_examples.savefig(path, dpi=100, bbox_inches="tight", facecolor="white")
print("wrote", path.resolve())